In [1]:
# Install required packages (if necessary)
!pip install torch torchvision scikit-learn pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import models, transforms
from sklearn.metrics import classification_report
from PIL import Image
import numpy as np
from google.colab import drive

In [6]:
drive.mount('/content/drive')  # Mount Google Drive

data_dir = "/content/drive/MyDrive/mentoring/Kelas-CV/faces_data"  # Update with your dataset path

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# Define custom dataset
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = []

        class_to_idx = {}

        for file_name in os.listdir(root_dir):
            if file_name.endswith(('.jpg', '.jpeg', '.png')):
                class_label = '_'.join(file_name.split('_')[:-1])
                if class_label not in class_to_idx:
                    class_to_idx[class_label] = len(class_to_idx)
                self.image_paths.append(os.path.join(root_dir, file_name))
                self.labels.append(class_to_idx[class_label])

        self.classes = list(class_to_idx.keys())
        self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
# Define data transforms
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset
dataset = FaceDataset(root_dir=data_dir, transform=data_transforms)

In [9]:
# Split dataset into training and test sets
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [10]:
# Load EfficientNetB0 model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
num_classes = len(dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 73.9MB/s]


In [11]:
def train_model(model, criterion, optimizer, num_epochs=10):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_preds = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            correct_preds += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_dataset)
        epoch_acc = correct_preds.double() / len(train_dataset)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

    print('Training complete')
    return model

In [12]:
# Train the model
model = train_model(model, criterion, optimizer)

Epoch 1/10, Loss: 1.3944, Accuracy: 0.6137
Epoch 2/10, Loss: 0.2598, Accuracy: 0.9299
Epoch 3/10, Loss: 0.1510, Accuracy: 0.9607
Epoch 4/10, Loss: 0.0892, Accuracy: 0.9744
Epoch 5/10, Loss: 0.0730, Accuracy: 0.9778
Epoch 6/10, Loss: 0.0540, Accuracy: 0.9786
Epoch 7/10, Loss: 0.0517, Accuracy: 0.9846
Epoch 8/10, Loss: 0.0722, Accuracy: 0.9786
Epoch 9/10, Loss: 0.0849, Accuracy: 0.9803
Epoch 10/10, Loss: 0.0851, Accuracy: 0.9786
Training complete


In [13]:
# Save the model to Google Drive
torch.save(model.state_dict(), '/content/drive/MyDrive/mentoring/Kelas-CV/Pertemuan9/full_classification_model.pth')

# Save the backbone model
backbone = nn.Sequential(*list(model.children())[:-1])
torch.save(backbone.state_dict(), '/content/drive/MyDrive/mentoring/Kelas-CV/Pertemuan9/backbone_model.pth')

# Evaluate model
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print classification report
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=[str(cls) for cls in dataset.classes]))

Classification Report:
                   precision    recall  f1-score   support

   Hrithik Roshan       0.82      0.90      0.86        10
        Zac Efron       0.90      1.00      0.95         9
          Kashyap       1.00      1.00      1.00         3
   Dwayne Johnson       1.00      1.00      1.00         5
     Henry Cavill       1.00      1.00      1.00        17
      Virat Kohli       1.00      1.00      1.00         2
        Brad Pitt       1.00      0.92      0.96        12
     Andy Samberg       1.00      0.86      0.92        14
     Hugh Jackman       1.00      1.00      1.00         7
           Marmik       0.50      1.00      0.67         2
Vijay Deverakonda       0.88      0.78      0.82         9
 Robert Downey Jr       1.00      1.00      1.00         6
       Tom Cruise       1.00      1.00      1.00         8
    Roger Federer       1.00      1.00      1.00        11
 Amitabh Bachchan       1.00      1.00      1.00        11
     Akshay Kumar       1.00    